# 🕵️ Dia 38 — Rastreando atores pelo que é distintivo neles

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 38* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-38.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Na reunião com as cinco cooperativas, a CredMinas comemorou o bloqueio do 203.0.113.77, só que o kit dela estava em outro IP. Rafa: *"Alugar outro servidor leva dez minutos. Mudar o golpe inteiro, não. E o `// pescador` no JavaScript ele apagaria em um segundo, mas não apagou em seis dias. Separa o que é caro do que é barato para ele, e o que só ele faz do que todo golpista faz."*

Neste notebook você vai:

1. **Refazer o `baserate.jq` em Python** e gerar `apt29-baserate.json`.
2. **Medir a sobreposição APT29 × APT28** pelos relacionamentos diretos.
3. **Jogar o quiz da Pirâmide da Dor.**
4. **Classificar as 18 observações do CA-ISCA-01** e achar o que sobreviveu à troca de infraestrutura.
5. **Desenhar a linha do tempo do cluster** em gantt Mermaid.

🐍 Só biblioteca padrão do Python. Nada para instalar. Funciona offline: toda célula que tenta a rede tem plano B embutido. Rode as células de cima para baixo (Shift+Enter).

## 1. A taxa de base (passo 1 do original)

O `baserate.jq` conta, para cada técnica, **quantos grupos do ATT&CK** têm um relacionamento `uses` com ela. Pouco grupo = mais distintivo.

A célula tenta baixar o ATT&CK completo e calcular tudo do zero (como o original). Sem rede, usa as **contagens da v19.2 embutidas** (calculadas do arquivo oficial em 2026-10-12, só para as técnicas que usamos hoje).

In [ ]:
#@title Fonte { display-mode: "form" }
BAIXAR_COMPLETO = False  #@param {type:"boolean"}

import collections      # Counter conta ocorrências
import json
import urllib.request

# Plano B: contagens v19.2 (técnica -> nº de grupos com 'uses' direto)
FREQ_V19_2 = {"T1001.001": 1, "T1003": 13, "T1003.001": 44, "T1003.002": 14, "T1003.003": 18, "T1003.004": 10, "T1005": 46, "T1014": 6, "T1016.001": 11, "T1021.002": 27, "T1021.007": 3, "T1025": 4, "T1027.001": 8, "T1027.002": 23, "T1027.006": 1, "T1027.013": 40, "T1030": 5, "T1036": 20, "T1036.005": 63, "T1037": 4, "T1037.001": 2, "T1037.004": 3, "T1039": 8, "T1040": 8, "T1047": 42, "T1048.002": 4, "T1053.005": 54, "T1056.001": 26, "T1056.003": 2, "T1057": 41, "T1059.001": 85, "T1059.003": 73, "T1059.006": 19, "T1059.009": 4, "T1068": 22, "T1070.004": 47, "T1070.006": 11, "T1071.001": 57, "T1071.003": 6, "T1074.001": 28, "T1074.002": 11, "T1078": 47, "T1078.003": 12, "T1078.004": 12, "T1083": 52, "T1087.004": 2, "T1090.002": 11, "T1090.003": 12, "T1090.004": 1, "T1091": 8, "T1092": 1, "T1098.002": 3, "T1098.005": 1, "T1102.002": 16, "T1105": 88, "T1110": 16, "T1110.001": 3, "T1110.003": 11, "T1113": 19, "T1114.002": 13, "T1119": 21, "T1120": 9, "T1133": 28, "T1134.001": 2, "T1136.003": 2, "T1137.002": 1, "T1140": 38, "T1189": 31, "T1190": 46, "T1199": 12, "T1203": 42, "T1204.001": 49, "T1204.002": 86, "T1210": 12, "T1211": 2, "T1213": 1, "T1213.002": 7, "T1218.005": 17, "T1218.011": 26, "T1221": 8, "T1498": 1, "T1505.003": 31, "T1528": 4, "T1542.003": 3, "T1546.003": 10, "T1546.008": 6, "T1546.015": 1, "T1547.001": 57, "T1548.002": 11, "T1550.001": 4, "T1550.002": 11, "T1550.003": 3, "T1553.005": 3, "T1556.007": 1, "T1557.004": 1, "T1559.002": 11, "T1560": 13, "T1560.001": 39, "T1564.001": 13, "T1564.003": 18, "T1566.001": 78, "T1566.002": 46, "T1566.003": 14, "T1567": 5, "T1568": 8, "T1573": 4, "T1573.001": 14, "T1583.001": 46, "T1583.003": 16, "T1583.006": 27, "T1584.008": 4, "T1585.001": 19, "T1586.002": 14, "T1586.003": 2, "T1587.001": 26, "T1587.003": 4, "T1588.002": 82, "T1588.004": 7, "T1588.007": 3, "T1589.001": 6, "T1591": 7, "T1595.002": 15, "T1596": 2, "T1598": 6, "T1598.003": 16, "T1608.005": 3, "T1621": 3, "T1649": 1, "T1651": 2, "T1657": 17, "T1665": 2, "T1669": 1, "T1684.001": 16, "T1685.002": 1, "T1685.005": 14}
APT29_V19_2 = ["T1003.002", "T1003.004", "T1005", "T1016.001", "T1021.007", "T1027.001", "T1027.002", "T1027.006", "T1036.005", "T1037", "T1037.004", "T1047", "T1053.005", "T1059.001", "T1059.006", "T1059.009", "T1068", "T1070.004", "T1070.006", "T1078", "T1078.003", "T1078.004", "T1087.004", "T1090.002", "T1090.003", "T1090.004", "T1098.002", "T1098.005", "T1105", "T1110.001", "T1110.003", "T1114.002", "T1133", "T1136.003", "T1190", "T1199", "T1203", "T1204.001", "T1204.002", "T1218.005", "T1505.003", "T1528", "T1546.003", "T1546.008", "T1547.001", "T1548.002", "T1550.003", "T1553.005", "T1556.007", "T1566.001", "T1566.002", "T1566.003", "T1568", "T1573", "T1583.006", "T1586.002", "T1586.003", "T1587.001", "T1587.003", "T1588.002", "T1595.002", "T1621", "T1649", "T1651", "T1665", "T1685.002"]
APT28_V19_2 = ["T1001.001", "T1003", "T1003.001", "T1003.003", "T1005", "T1014", "T1021.002", "T1025", "T1027.013", "T1030", "T1036", "T1036.005", "T1037.001", "T1039", "T1040", "T1048.002", "T1056.001", "T1057", "T1059.001", "T1059.003", "T1068", "T1070.004", "T1070.006", "T1071.001", "T1071.003", "T1074.001", "T1074.002", "T1078", "T1078.004", "T1083", "T1090.002", "T1090.003", "T1091", "T1092", "T1098.002", "T1102.002", "T1105", "T1110", "T1110.001", "T1110.003", "T1113", "T1114.002", "T1119", "T1120", "T1133", "T1134.001", "T1137.002", "T1140", "T1189", "T1190", "T1199", "T1203", "T1204.001", "T1204.002", "T1210", "T1211", "T1213", "T1213.002", "T1218.011", "T1221", "T1498", "T1505.003", "T1528", "T1542.003", "T1546.015", "T1547.001", "T1550.001", "T1550.002", "T1557.004", "T1559.002", "T1560", "T1560.001", "T1564.001", "T1564.003", "T1566.001", "T1567", "T1573.001", "T1583.001", "T1583.003", "T1583.006", "T1584.008", "T1586.002", "T1588.002", "T1588.007", "T1589.001", "T1591", "T1595.002", "T1596", "T1598", "T1598.003", "T1669", "T1684.001", "T1685.005"]

def calcular_do_zero(objs, grupos=("APT29", "APT28")):
    """O baserate.jq do original, em Python. Devolve (freq, {grupo: técnicas})."""
    # 1) mapa id STIX da técnica -> T####
    tid = {o["id"]: o["external_references"][0]["external_id"] for o in objs if o["type"] == "attack-pattern"}
    # 2) mapa nome do grupo -> id STIX
    gid = {o["name"]: o["id"] for o in objs if o["type"] == "intrusion-set"}
    # 3) pares únicos (grupo, técnica) com 'uses' não revogado
    usos = {(r["source_ref"], tid[r["target_ref"]]) for r in objs
            if r["type"] == "relationship" and r.get("relationship_type") == "uses" and not r.get("revoked")
            and r["source_ref"].startswith("intrusion-set") and r["target_ref"].startswith("attack-pattern")}
    freq = collections.Counter(t for _, t in usos)             # 4) técnica -> nº de grupos
    por_grupo = {g: sorted({t for s, t in usos if s == gid[g]}) for g in grupos}
    return dict(freq), por_grupo

FREQ, APT29, APT28, fonte = FREQ_V19_2, APT29_V19_2, APT28_V19_2, "contagens v19.2 embutidas (2026-10-12)"
if BAIXAR_COMPLETO:
    try:
        url = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json"
        with urllib.request.urlopen(url, timeout=120) as r:
            objs = json.load(r)["objects"]
        FREQ, pg = calcular_do_zero(objs)
        APT29, APT28 = pg["APT29"], pg["APT28"]
        fonte = "ATT&CK completo pela rede"
    except Exception as erro:
        print(f"⚠️ Sem rede ({erro.__class__.__name__}). Usando as contagens embutidas.")
print("Fonte:", fonte)

camada = {
    "name": "APT29: how many ATT&CK groups share each technique",
    "versions": {"layer": "4.5", "attack": "19", "navigator": "5.3.2"},
    "domain": "enterprise-attack",
    "description": "Score = number of groups in ATT&CK with a uses relationship to the technique. Low = more distinctive.",
    "techniques": [{"techniqueID": t, "score": FREQ[t], "comment": f"{FREQ[t]} groups use this"} for t in APT29],
    "gradient": {"colors": ["#1a9850", "#fee08b", "#d73027"], "minValue": 1, "maxValue": 60},
}
with open("apt29-baserate.json", "w") as f:
    json.dump(camada, f, indent=1)

ordenado = sorted(camada["techniques"], key=lambda t: (t["score"], t["techniqueID"]))
print("\nAs 6 mais raras (o original viu 6 técnicas com 1 grupo):")
for t in ordenado[:6]:
    print(f"  {t['score']}\t{t['techniqueID']}")
print("\nAs 3 mais comuns:")
for t in ordenado[-3:]:
    print(f"  {t['score']}\t{t['techniqueID']}")
print("\n💾 apt29-baserate.json (verde = distintivo, vermelho = comum)")

## 2. Sobreposição APT29 × APT28 (passos 2 e 3)

Aqui contamos pelos **relacionamentos diretos** do dataset. O original avisa: as camadas baixadas do site incluem técnicas de **campanhas** atribuídas aos grupos e dão outro número (47 contra 29 na v19.2). Anote sempre qual definição usou.

In [ ]:
a, b = set(APT29), set(APT28)
compartilhadas = sorted(a & b, key=lambda t: -FREQ[t])   # as mais comuns primeiro

print(f"Só APT29: {len(a - b)} | Só APT28: {len(b - a)} | Os dois (pontuação 3): {len(compartilhadas)}")
print("(Definição: relacionamentos diretos grupo → técnica. Na v19.2, 29.)\n")
print("| Técnica | Grupos no ATT&CK | Distingue os dois? |\n|---|---|---|")
for t in compartilhadas:
    print(f"| {t} | {FREQ[t]} | não (os dois usam) |")
print("\nSe um incidente mostrasse só essas técnicas, o que daria para dizer sobre qual dos dois? Nada.")

## 3. 🎮 Quiz: em que nível da Pirâmide está?

Responda **1** (hash), **2** (IP), **3** (domínio), **4** (artefato de rede/host), **5** (ferramenta) ou **6** (TTP).

In [ ]:
MODO_JOGO = True

NIVEIS = {"1": "Hash", "2": "IP", "3": "Domínio", "4": "Artefato de rede/host", "5": "Ferramenta", "6": "TTP"}
QUIZ = [
    ("SHA-256 do isca.js da Serra Azul", "1", "Recompilou, mudou: aconteceu entre a Serra Azul e a CredMinas."),
    ("203.0.113.77", "2", "Alugar outro VPS leva minutos."),
    ("coopvale-pix.example", "3", "Registrar outro custa pouco."),
    ("Comentário '// pescador' dentro do JavaScript do kit", "4", "Artefato de host. Barato de tirar, mas ninguém tirou: é sinal."),
    ("Favicon customizado do IscaKit", "4", "Artefato de rede: aparece igual em domínios diferentes."),
    ("O próprio IscaKit (o kit vendido)", "5", "Ferramenta. Mas COMPARTILHADA: outros compradores também usam."),
    ("Registrar domínio poucos dias antes, pegar certificado na véspera e disparar SMS com link encurtado", "6",
     "É o modo de operar. O mais caro de mudar."),
    ("Mostrar página em branco para IPs da cooperativa e da Vigia (cloaking)", "6", "Comportamento. Apareceu em 6/out."),
]
acertos = 0
for i, (item, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {item}")
    if MODO_JOGO:
        r = ""
        while r not in NIVEIS:
            r = input("   1=hash 2=IP 3=domínio 4=artefato 5=ferramenta 6=TTP: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end=" ")
    print(f"{NIVEIS[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 4. ➕ As 18 observações do CA-ISCA-01

O CSV `caso-continuo/dados/dia-38/observacoes-ca-isca-01.csv` está embutido abaixo. Primeiro, a contagem por nível.

In [ ]:
import csv   # leitor de CSV da biblioteca padrão
import io    # para ler um texto como se fosse arquivo

CSV_OBS = '''# Observações datadas do cluster CA-ISCA-01 — 100% FICTÍCIO (Operação Isca, Investigador Cyber 90 Dias, Dia 38).
# Domínios .example (RFC 2606), IPs de documentação (RFC 5737). Fontes F1-F6: ver caso-continuo/dados/dia-37/fontes-ca-isca-01.json
data_atividade,cooperativa_alvo,indicador_ou_comportamento,tipo,nivel_piramide,fonte
2026-10-02,Serra Azul,serraazul-pix.example registrado (Registrar Barato),dominio,Domínio,F1
2026-10-04,Serra Azul,certificado gratuito com SAN serraazul-pix + serraazul-seguranca,certificado,Artefato de rede/host,F1
2026-10-05,Serra Azul,SMS 'chave Pix bloqueada' com lnk.example/sa7,comportamento,TTP,F2
2026-10-05,Serra Azul,serraazul-pix.example em 203.0.113.77,ip,IP,F1
2026-10-05,Serra Azul,isca.js com comentário '// pescador' (SHA-256 c6a500d4...),hash,Hash,F1
2026-10-05,Serra Azul,favicon customizado do IscaKit (hash Shodan -1029384756),artefato,Artefato de rede/host,F1
2026-10-05,Serra Azul,IscaKit (kit vendido em grupo de mensagens),ferramenta,Ferramenta,F5
2026-10-06,Serra Azul,kit passa a servir página em branco a IPs da cooperativa e da Vigia (cloaking por origem),comportamento,TTP,F1
2026-10-07,CoopVale,coopvale-pix.example registrado,dominio,Domínio,F3
2026-10-08,CoopVale,SMS 'chave Pix bloqueada' com link para coopvale-pix.example,comportamento,TTP,F3
2026-10-08,CoopVale,coopvale-pix.example em 203.0.113.77 (mesmo IP da Serra Azul),ip,IP,F3
2026-10-09,CredMinas,credminas-seguranca.example em 198.51.100.40,ip,IP,F2
2026-10-09,CredMinas,mesmo favicon customizado (hash -1029384756),artefato,Artefato de rede/host,F2
2026-10-09,CredMinas,isca.js com novo SHA-256 (comentário '// pescador' mantido),hash,Hash,F2
2026-10-10,Unicoop Sul,unicoop-sul-pix.example com SAN compartilhado com credminas-seguranca,certificado,Artefato de rede/host,F2
2026-10-10,Unicoop Sul,SMS 'chave Pix bloqueada',comportamento,TTP,F2
2026-10-11,Cooperouro,cooperouro-regulariza.example só com o mesmo título genérico,dominio,Domínio,F2
2026-10-11,todas,perfis falsos em rede social respondendo reclamações com o link,comportamento,TTP,F2
'''

linhas = [l for l in CSV_OBS.splitlines() if l and not l.startswith("#")]   # tira os comentários
OBS = list(csv.DictReader(io.StringIO("\n".join(linhas))))
print(f"{len(OBS)} observações.\n")

ORDEM = ["Hash", "IP", "Domínio", "Artefato de rede/host", "Ferramenta", "TTP"]
cont = collections.Counter(o["nivel_piramide"] for o in OBS)
for nivel in reversed(ORDEM):                               # desenha a pirâmide de cima para baixo
    print(f"{nivel:<22} {'█' * cont[nivel]} {cont[nivel]}")
baixo = sum(cont[n] for n in ORDEM[:3])
print(f"\n{baixo} de {len(OBS)} nos três níveis de baixo (hash, IP, domínio): é o que vira lista de bloqueio, e o que envelhece primeiro.")

### O que sobreviveu à troca de infraestrutura?

Compare a Serra Azul (5/out) com a CredMinas (9/out). Mudou o IP. Mudou o hash. O que ficou igual?

In [ ]:
def da(coop):
    return {o["indicador_ou_comportamento"]: o for o in OBS if o["cooperativa_alvo"] == coop}

sa, cm = da("Serra Azul"), da("CredMinas")
print("Serra Azul:"); [print("  -", k) for k in sa]
print("CredMinas:");  [print("  -", k) for k in cm]

# O que se repete: procuramos palavras-chave que aparecem nos dois lados
for chave in ["favicon customizado", "// pescador", "203.0.113.77", "SHA-256"]:
    lado_a = any(chave in k for k in sa)
    lado_b = any(chave in k for k in cm)
    print(f"{chave:<22} Serra Azul: {'sim' if lado_a else 'não':<4} CredMinas: {'sim' if lado_b else 'não'}")
print("\n→ Sobreviveram: o FAVICON (artefato) e a string '// pescador' (artefato), apesar do novo hash do arquivo.")

### Taxa de base do vocabulário do IscaKit

Lembrete do original: *"uma técnica que parece rara no ATT&CK pode simplesmente ser pouco relatada"*. O ATT&CK quase não cobre crime financeiro brasileiro por SMS.

In [ ]:
VOCAB = {"T1566.002": "SMS/link", "T1583.001": "domínios imitação", "T1078": "login com senha capturada",
         "T1588.004": "certificado obtido", "T1608.005": "encurtador preparado", "T1056.003": "captura no portal falso",
         "T1657": "Pix (roubo financeiro)", "T1684.001": "se passar pela cooperativa"}
for t, desc in sorted(VOCAB.items(), key=lambda x: FREQ.get(x[0], 0)):
    n = FREQ.get(t, 0)
    leitura = "comum: contexto" if n >= 15 else ("incomum NO ATT&CK (pouco relatado?)" if n <= 5 else "intermediário")
    print(f"{t:<10} {n:>3} grupos  {desc:<28} → {leitura}")

In [ ]:
#@title Suas notas de citação { display-mode: "form" }
nunca_citaria_1 = "Texto 'Pix bloqueado' no SMS: tipo de golpe comum no Brasil"  #@param {type:"string"}
nunca_citaria_2 = "T1566.002 (46 grupos no ATT&CK v19.2)"  #@param {type:"string"}
nunca_citaria_3 = "IP 203.0.113.77: hospedagem trocada em 4 dias, 41 domínios no mesmo IP"  #@param {type:"string"}
sinal = "Favicon customizado (hash -1029384756, 9 resultados) + string '// pescador' no JS"  #@param {type:"string"}
o_que_falta = "Um segundo pivô independente (certificado ou padrão de registro) no mesmo domínio novo, e conferir que o kit não é vendido com esse favicon padrão"  #@param {type:"string"}

notas = f"""# Notas do Dia 38 — CA-ISCA-01 (ATT&CK v19.2, contagens: {fonte})
## Nunca citaria como evidência de que um SMS novo é do CA-ISCA-01
1. {nunca_citaria_1}
2. {nunca_citaria_2}
3. {nunca_citaria_3}
## Sinal significativo (não conclusivo)
{sinal}
Falta para subir a confiança: {o_que_falta}
## Sobreposição APT29 × APT28
{len(compartilhadas)} técnicas (definição: relacionamentos diretos; camadas baixadas dão outro número)
"""
with open("notas-dia38.md", "w", encoding="utf-8") as f:
    f.write(notas)
print(notas)

## 5. Linha do tempo do cluster (passo 4, versão Isca)

O gantt do original, com duas faixas: **infraestrutura** (o que muda rápido) e **comportamento** (o que muda devagar). Cole a saída em [mermaid.live](https://mermaid.live/) ou num `.md` no GitHub.

In [ ]:
FAIXA = {"TTP": "Comportamento", "Ferramenta": "Comportamento", "Artefato de rede/host": "Artefatos",
         "Hash": "Infraestrutura", "IP": "Infraestrutura", "Domínio": "Infraestrutura"}
gantt = ["```mermaid", "gantt", "    title CA-ISCA-01 por data da atividade (fictício)",
         "    dateFormat YYYY-MM-DD", "    axisFormat %d/%m"]
for faixa in ["Infraestrutura", "Artefatos", "Comportamento"]:
    gantt.append(f"    section {faixa}")
    for o in OBS:
        if FAIXA[o["nivel_piramide"]] == faixa:
            rotulo = (o["cooperativa_alvo"] + ": " + o["indicador_ou_comportamento"])[:55].replace(":", " -")
            gantt.append(f"    {rotulo} :milestone, {o['data_atividade']}, 0d")
gantt.append("```")
texto = "\n".join(gantt)
print(texto)

antes = [o["data_atividade"] for o in OBS if o["nivel_piramide"] == "TTP" and "cloaking" not in o["indicador_ou_comportamento"]]
cloak = [o["data_atividade"] for o in OBS if "cloaking" in o["indicador_ou_comportamento"]]
print(f"\nSem cloaking registrado: {min(antes)} (primeiro SMS); cloaking aparece em: {cloak[0]}.")

with open("linha-do-tempo-ca-isca-01.md", "w", encoding="utf-8") as f:
    f.write("# Linha do tempo — CA-ISCA-01\n\n" + texto + "\n")
try:
    from google.colab import files
    for nome in ["apt29-baserate.json", "notas-dia38.md", "linha-do-tempo-ca-isca-01.md"]:
        files.download(nome)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## ✅ Antes de fechar o Dia 38

- [ ] `apt29-baserate.json` no Navigator, exportado em SVG
- [ ] Sobreposição APT29 × APT28 anotada **com a definição usada**
- [ ] Pirâmide do CA-ISCA-01 contada, e o que sobreviveu à troca de IP nomeado
- [ ] Três "nunca citaria" + um sinal significativo, com o que falta
- [ ] Linha do tempo com a data do cloaking

**Amanhã (Dia 39):** enriquecer IOCs **sem tocar no adversário**. Rafa vai conferir seu histórico de shell atrás de `dig` e `ping`.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>